# 04 Snow reliability

Defines elevation bands from the master plan's own stated elevations (claim C033) and records the one historical snowfall figure available (claim C034) as the baseline a reliability indicator would be validated against. It does NOT compute a reliability indicator against baseline/mid-century climate projections: no CanDCS-M6/SWE grid cell for Revelstoke has been pulled yet (see Open issues in steps/04). Producing a plausible-looking projection number without that data would violate CLAUDE.md's rule against stating a figure with no evidence behind it, so this notebook stops at what it can actually show.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
project_crs = "EPSG:26911"
dem_clip_buffer_m = 200
aspect_class_schemes = [4, 8]
baseline_horizon = "1991-2020"
mid_century_horizon = "2041-2070"
ssp_scenarios = ["ssp245", "ssp585"]
snow_to_water_ratios = [8, 10, 13]  # cm of snow depth per 10mm SWE; judgment call, not sourced for Revelstoke
climate_grid_footprint_km = (9, 6)  # approx. grid spacing at this latitude, for the elevation-gap report only

## Load the ledger

Confirms C033 (elevation) and C034 (historical snowfall) still point to the source this notebook reads from.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C033"]["source_id"] == "S034"
assert ledger["claims"]["C034"]["source_id"] == "S034"
print("C033, C034 confirmed against S034")

## Elevation bands (claim C033)

The master plan states five elevation points directly (bottom, lift top, sub peak, summit) rather than a continuous profile, so bands are built between those stated points instead of picking arbitrary round-number cutoffs. This also gives a free internal consistency check: lift top minus bottom should equal the plan's separately stated lift-accessed vertical.

In [ ]:
import pandas as pd

elevation_points_m = {
    "Bottom (Lower Village)": 512,
    "Lift top (The Stoke Chair)": 2225,
    "Sub Peak": 2340,
    "Mt. Mackenzie Summit": 2466,
}
bands = pd.DataFrame(
    [
        {"band": "Base to lift top", "bottom_m": 512, "top_m": 2225},
        {"band": "Lift top to sub peak", "bottom_m": 2225, "top_m": 2340},
        {"band": "Sub peak to summit", "bottom_m": 2340, "top_m": 2466},
    ]
)
bands["band_vertical_m"] = bands["top_m"] - bands["bottom_m"]

stated_lift_accessed_vertical_m = 1713
assert bands.loc[0, "band_vertical_m"] == stated_lift_accessed_vertical_m, (
    "base-to-lift-top band should match the plan's stated lift-accessed vertical"
)
bands

## Historical snowfall baseline (claim C034)

The master plan states a range, not a single figure, and no year range or measurement method for it. Both bounds are kept, and the uncertainty is written into the output rather than silently collapsed to a midpoint.

In [ ]:
historical_snowfall = {
    "rmr_annual_m_low": 9,
    "rmr_annual_m_high": 14,
    "selkirk_mountains_annual_m_low": 12,
    "selkirk_mountains_annual_m_high": 18,
}
historical_snowfall

## Reliability indicator (judgment call; evaluated further below)

Proposes the indicator as a ratio: projected seasonal snowfall divided by the historical baseline above. This is a judgment call (no claim backs this specific formula). Real projected snowfall is now available (C064), so this is evaluated later in this notebook -- but not directly here, since the projection is in mm of snow water equivalent (SWE) and the historical baseline above is in metres of snow depth, units that are not the same thing without a density assumption. See the "Snow-to-water density" section below for how that is handled.

In [ ]:
def snow_reliability_ratio(projected_seasonal_snowfall_m, historical_baseline_m):
    """Ratio of projected to historical seasonal snowfall. >1 means
    snowier than the historical baseline, <1 means less snowy. Evaluated
    further below, across a sensitivity range, not with a single default."""
    return projected_seasonal_snowfall_m / historical_baseline_m

## Write outputs

Elevation bands and the historical baseline are real outputs; the indicator function is recorded as a method, not a result, so nothing computed from it goes to data/processed yet.

In [ ]:
import json
import os

os.makedirs(processed_dir, exist_ok=True)
bands.to_csv(f"{processed_dir}/04_elevation_bands.csv", index=False, encoding="utf-8")
with open(f"{processed_dir}/04_historical_snowfall.json", "w", encoding="utf-8") as f:
    json.dump(historical_snowfall, f, indent=2)
print("wrote 04_elevation_bands.csv, 04_historical_snowfall.json")

## Checks

The band boundaries must be strictly increasing and the base band's vertical must match the plan's separately stated lift-accessed vertical (already asserted above); this check also records, in the output itself, that no projection has run yet, so a downstream step cannot mistake an empty result for a zero.

In [ ]:
assert (bands["top_m"] > bands["bottom_m"]).all()
assert bands["bottom_m"].is_monotonic_increasing
assert historical_snowfall["rmr_annual_m_low"] < historical_snowfall["rmr_annual_m_high"]
print("checks passed (elevation bands and historical baseline)")

# Spatial + climate addition (steps/04's proposed method)

Replaces the 3-row elevation-band table with real area and run length by elevation band x aspect class, and pulls real projected snowfall now that the CanDCS-M6 access blocker is resolved (C064). Stops short of a snow-adjusted capacity number: that needs a lapse-rate adjustment this step does not attempt (see Open issues in steps/04).

## Confirm the claims this section rests on

C064 is the resolved access path; C035 is the earlier claim it supersedes (the base archive does contain a snowfall variable, contrary to what S016's own descriptive page implied).

In [ ]:
sys.path.insert(0, "src")
from resort.dem import aspect_class, clip_dem, compute_slope_aspect, sample_at_point

assert ledger["claims"]["C064"]["source_id"] == "S015"
assert ledger["claims"]["C035"]["conflicts_with"] == "C064"
# C064 verifies the simulations_30yAvg (per-GCM) path; the cell below actually reads the
# sibling ensemble_percentiles (multi-model blend) path instead, which needed its own claim
# (caught by a fresh Phase-3 notebook-reviewer pass -- the two paths are real and both work,
# but citing C064 alone for a different product it didn't test was a real citation gap).
assert ledger["claims"]["C075"]["source_id"] == "S015"
print("C064/C035/C075 confirmed")

## Load the AOI and runs from step 03

Reuses step 03's own outputs rather than re-deriving the AOI or re-querying OSM.

In [ ]:
import geopandas as gpd

aoi = gpd.read_file(f"{processed_dir}/03_aoi.gpkg")
runs = gpd.read_file(f"{processed_dir}/03_runs.gpkg")
print(f"AOI: {aoi.crs}, {len(runs)} runs loaded")

## DEM: slope and aspect over the AOI + runs extent

Same windowed-read approach as step 03 (now shared via src/resort/dem.py), clipped to the union of the AOI and run bounds so no run's points fall outside the window.

In [ ]:
import numpy as np

combined_bounds = np.vstack([aoi.total_bounds, runs.total_bounds])
bounds = (combined_bounds[:, 0].min(), combined_bounds[:, 1].min(),
          combined_bounds[:, 2].max(), combined_bounds[:, 3].max())

dem = clip_dem(bounds, project_crs, buffer_m=dem_clip_buffer_m)
slope_deg, slope_pct, aspect_deg = compute_slope_aspect(dem)
transform_affine = dem.rio.transform()
dem_arr = dem.values[0]
pixel_size = dem.rio.resolution()
print(f"DEM shape {dem_arr.shape}, CRS {dem.rio.crs}, pixel size {pixel_size}")
assert abs(abs(pixel_size[0]) - 1.0) < 0.1

## Elevation-band x aspect area

For each pixel inside the AOI, classify by the same elevation bands already written to data/processed/04_elevation_bands.csv (from the plan's own stated points, C033) and by aspect, under both 4-way and 8-way compass schemes -- reported side by side as a sensitivity check on the binning choice, not a single silently-picked scheme.

In [ ]:
import pandas as pd
from rasterio.features import geometry_mask

aoi_mask = ~geometry_mask(
    [aoi.geometry.iloc[0]], out_shape=dem_arr.shape, transform=transform_affine, invert=False
)
in_aoi = aoi_mask & ~np.isnan(dem_arr)

bands_check = pd.read_csv(f"{processed_dir}/04_elevation_bands.csv")
assert bands_check["bottom_m"].tolist() == bands["bottom_m"].tolist()
assert bands_check["top_m"].tolist() == bands["top_m"].tolist()
band_edges = bands["bottom_m"].tolist() + [bands["top_m"].iloc[-1]]
band_labels = bands["band"].tolist()

elev_band_idx = np.digitize(dem_arr, band_edges) - 1
elev_band_idx = np.clip(elev_band_idx, 0, len(band_labels) - 1)

rows = []
for n_classes in aspect_class_schemes:
    aspect_labels = np.vectorize(lambda a: aspect_class(a, n_classes))(aspect_deg[in_aoi])
    bands_in_aoi = np.array(band_labels)[elev_band_idx[in_aoi]]
    counts = pd.Series(zip(bands_in_aoi, aspect_labels)).value_counts()
    pixel_area_m2 = abs(pixel_size[0] * pixel_size[1])
    for (band, aspect), count in counts.items():
        rows.append({
            "aspect_scheme": f"{n_classes}-way", "band": band, "aspect": aspect,
            "area_ha": round(count * pixel_area_m2 / 10_000, 2),
        })
elevation_aspect_bands = pd.DataFrame(rows)
elevation_aspect_bands.head(10)

## Run length by elevation band

Each run is assigned to the band containing its mean sampled elevation (a simplification: a run spanning a band boundary is counted once, not split proportionally). Documented here as that specific simplification, not left implicit.

In [ ]:
run_band_rows = []
for _, run in runs.iterrows():
    elevs = np.array([sample_at_point(dem_arr, x, y, transform_affine) for x, y in run.geometry.coords])
    mean_elev = np.nanmean(elevs)
    band_idx = int(np.clip(np.digitize(mean_elev, band_edges) - 1, 0, len(band_labels) - 1))
    run_band_rows.append({"osm_id": run.get("osm_id"), "band": band_labels[band_idx], "length_m": run.geometry.length})

run_length_by_band = (
    pd.DataFrame(run_band_rows).groupby("band")["length_m"].sum().reindex(band_labels, fill_value=0.0).reset_index()
)
run_length_by_band["length_km"] = (run_length_by_band["length_m"] / 1000).round(2)
run_length_by_band

## Real projected snowfall (C075)

Pulls the multi-model ensemble percentile product (not a single GCM) for each SSP, at RMR's nearest grid cell, via OPeNDAP subsetting, confirmed live and fast (a couple of seconds, not a full-file download). C064 verifies a sibling per-GCM path (simulations_30yAvg); this ensemble_percentiles path is a different, separately-cited product (C075). If this fails when actually run, the exception is allowed to surface with its real message rather than being caught and hidden, per the user's own instruction to record exactly what failed.

In [ ]:
import xarray as xr

aoi_centroid_4326 = gpd.GeoSeries([aoi.geometry.iloc[0].centroid], crs=project_crs).to_crs("EPSG:4326").iloc[0]
rmr_lat, rmr_lon = aoi_centroid_4326.y, aoi_centroid_4326.x
sntot_base_url = (
    "https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/birdhouse/disk3/"
    "cccs_portal/indices/Final/CanDCS-M6/sntot/YS/{ssp}/ensemble_percentiles/"
    "sntot_ann_MBCn+PCIC-Blend_historical+{ssp}_1951-2100_30ymean_percentiles.nc"
)

snow_projection_rows = []
grid_cell_latlon = None
for ssp in ssp_scenarios:
    ds = xr.open_dataset(sntot_base_url.format(ssp=ssp))
    cell = ds.sel(lat=rmr_lat, lon=rmr_lon, method="nearest")[["sntot_p10", "sntot_p50", "sntot_p90"]].load()
    grid_cell_latlon = (float(cell.lat), float(cell.lon))
    horizons = [h.decode() if isinstance(h, bytes) else h for h in cell.horizon.values]
    for horizon, p10, p50, p90 in zip(horizons, cell["sntot_p10"].values, cell["sntot_p50"].values, cell["sntot_p90"].values):
        if horizon in (baseline_horizon, mid_century_horizon):
            snow_projection_rows.append({
                "ssp": ssp, "horizon": horizon,
                "sntot_p10_mm": round(float(p10), 1), "sntot_p50_mm": round(float(p50), 1), "sntot_p90_mm": round(float(p90), 1),
            })
snow_projection = pd.DataFrame(snow_projection_rows)
snow_projection

## Grid-cell-vs-terrain elevation gap

The climate grid cell has no elevation of its own in this product; what is reportable is how much real terrain-elevation variation exists inside one coarse grid cell's footprint, versus the AOI's own, much narrower, elevation range. This is the gap the step's plan asked for, reported before any adjustment -- no lapse-rate correction is attempted.

In [ ]:
from shapely.geometry import box
from shapely.ops import transform as shp_transform
import pyproj

half_lat_deg = (climate_grid_footprint_km[0] / 111) / 2
half_lon_deg = (climate_grid_footprint_km[1] / (111 * np.cos(np.radians(rmr_lat)))) / 2
grid_cell_box_4326 = box(
    grid_cell_latlon[1] - half_lon_deg, grid_cell_latlon[0] - half_lat_deg,
    grid_cell_latlon[1] + half_lon_deg, grid_cell_latlon[0] + half_lat_deg,
)
grid_cell_box_project = shp_transform(
    pyproj.Transformer.from_crs("EPSG:4326", project_crs, always_xy=True).transform, grid_cell_box_4326
)
grid_mask = ~geometry_mask([grid_cell_box_project], out_shape=dem_arr.shape, transform=transform_affine, invert=False)
grid_elevs = dem_arr[grid_mask & ~np.isnan(dem_arr)]
aoi_elevs = dem_arr[in_aoi]

climate_grid_overlay = {
    "grid_cell_lat": grid_cell_latlon[0], "grid_cell_lon": grid_cell_latlon[1],
    "grid_footprint_km": list(climate_grid_footprint_km),
    "grid_cell_dem_min_m": round(float(np.nanmin(grid_elevs)), 1) if grid_elevs.size else None,
    "grid_cell_dem_max_m": round(float(np.nanmax(grid_elevs)), 1) if grid_elevs.size else None,
    "aoi_dem_min_m": round(float(np.nanmin(aoi_elevs)), 1),
    "aoi_dem_max_m": round(float(np.nanmax(aoi_elevs)), 1),
}
climate_grid_overlay

## Snow-to-water density: reliability indicator, sensitivity-tested

sntot is in mm of snow water equivalent; the historical baseline (C034) is in metres of snow depth. Converting between them needs a snow-to-water ratio, which is not sourced for Revelstoke specifically; run across three plausible values instead of picking one, per the project's rule that an unsourced parameter gets a sensitivity test, not a silent default.

The same ratios are also run against the 1991-2020 baseline horizon, not only mid-century. This is the most direct check available on the whole unit-conversion approach: the baseline period uses the same data source and the same ratio logic, with no climate-change trend involved, so if it lands far outside C034's stated 9-14 m range, that points to a problem with the ratio or the data rather than to a genuine future decline in the mid-century numbers below.

In [ ]:
reliability_rows = []
relevant_horizons = snow_projection["horizon"].isin([baseline_horizon, mid_century_horizon])
for _, proj_row in snow_projection[relevant_horizons].iterrows():
    for ratio in snow_to_water_ratios:
        projected_depth_m = proj_row["sntot_p50_mm"] * ratio / 1000
        for hist_label, hist_m in [
            ("rmr_low", historical_snowfall["rmr_annual_m_low"]),
            ("rmr_high", historical_snowfall["rmr_annual_m_high"]),
        ]:
            reliability_rows.append({
                "ssp": proj_row["ssp"], "horizon": proj_row["horizon"], "snow_to_water_ratio": ratio,
                "projected_depth_m": round(projected_depth_m, 2), "historical_baseline": hist_label,
                "historical_m": hist_m,
                "reliability_ratio": round(snow_reliability_ratio(projected_depth_m, hist_m), 2),
            })
reliability_sensitivity = pd.DataFrame(reliability_rows)
projections_available = True
reliability_sensitivity

### Baseline sanity check against C034

If the baseline row's reliability ratio is far from 1.0 at every tested ratio, the mismatch is in the unit conversion or the climate-data baseline period itself, not in a real mid-century trend -- reported here rather than left implicit.

In [ ]:
baseline_check = reliability_sensitivity[reliability_sensitivity["horizon"] == baseline_horizon]
baseline_ratio_range = (baseline_check["reliability_ratio"].min(), baseline_check["reliability_ratio"].max())
print(
    f"baseline ({baseline_horizon}) reliability ratio across all tested snow-to-water ratios "
    f"and both SSPs: {baseline_ratio_range[0]:.2f} to {baseline_ratio_range[1]:.2f} "
    f"(1.0 would mean the baseline climate data exactly reproduces C034 under that ratio)"
)
baseline_check

## Write outputs


In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
elevation_aspect_bands.to_csv(f"{processed_dir}/04_elevation_aspect_bands.csv", index=False, encoding="utf-8")
run_length_by_band.to_csv(f"{processed_dir}/04_run_length_by_band.csv", index=False, encoding="utf-8")
snow_projection.to_csv(f"{processed_dir}/04_snow_projection.csv", index=False, encoding="utf-8")
reliability_sensitivity.to_csv(f"{processed_dir}/04_reliability_sensitivity.csv", index=False, encoding="utf-8")
with open(f"{processed_dir}/04_climate_grid_overlay.json", "w", encoding="utf-8") as f:
    json.dump(climate_grid_overlay, f, indent=2)
print("wrote 04_elevation_aspect_bands.csv, 04_run_length_by_band.csv, 04_snow_projection.csv, "
      "04_reliability_sensitivity.csv, 04_climate_grid_overlay.json")

## Checks

Band areas (each aspect scheme, summed) must equal the AOI's own total area; run-length total must equal the sum of every run's own length; the climate pull produced real, non-null values for every requested SSP/horizon; `projections_available` is genuinely True now, not left stale.

In [ ]:
aoi_area_ha = aoi.geometry.area.iloc[0] / 10_000
for scheme in elevation_aspect_bands["aspect_scheme"].unique():
    scheme_total = elevation_aspect_bands.loc[elevation_aspect_bands["aspect_scheme"] == scheme, "area_ha"].sum()
    assert abs(scheme_total - aoi_area_ha) < 1, f"{scheme} band areas ({scheme_total:.1f} ha) don't sum to AOI area ({aoi_area_ha:.1f} ha)"

assert abs(run_length_by_band["length_m"].sum() - runs.geometry.length.sum()) < 1
assert snow_projection[["sntot_p10_mm", "sntot_p50_mm", "sntot_p90_mm"]].notna().all().all()
assert len(snow_projection) == len(ssp_scenarios) * 2, "expected baseline + mid-century for every SSP"
assert projections_available is True
assert (reliability_sensitivity["horizon"] == baseline_horizon).any(), (
    "baseline horizon missing from the reliability sensitivity table -- the C034 sanity check needs it"
)
print("checks passed")

## Map: elevation bands, runs, and the climate grid cell footprint

One static map, downsampled for display.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from matplotlib.patches import Patch

downsample = 5
dem_small = dem_arr[::downsample, ::downsample]
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(np.nan_to_num(dem_small, nan=np.nanmin(dem_small)), vert_exag=1.5)

band_colors = {band_labels[0]: "#a1d99b", band_labels[1]: "#fdae6b", band_labels[2]: "#de2d26"}
band_rgba = np.zeros((*dem_small.shape, 4))
for idx, label in enumerate(band_labels):
    band_rgba[(elev_band_idx[::downsample, ::downsample] == idx) & in_aoi[::downsample, ::downsample]] = (
        *plt.matplotlib.colors.to_rgb(band_colors[label]), 0.5
    )

fig, ax = plt.subplots(figsize=(9, 9))
extent = dem.rio.bounds()
ax.imshow(hillshade, cmap="gray", extent=(extent[0], extent[2], extent[1], extent[3]), origin="upper")
ax.imshow(band_rgba, extent=(extent[0], extent[2], extent[1], extent[3]), origin="upper")
aoi.boundary.plot(ax=ax, color="black", linewidth=1.5)
runs.plot(ax=ax, color="blue", linewidth=0.4)
gpd.GeoSeries([grid_cell_box_project]).boundary.plot(ax=ax, color="purple", linewidth=2, linestyle="--")
ax.set_title("Elevation bands, runs, and climate grid cell footprint (dashed)")
ax.set_xlabel(f"Easting ({project_crs})")
ax.set_ylabel("Northing")
legend_handles = [Patch(facecolor=c, alpha=0.5, label=b) for b, c in band_colors.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Versions

In [ ]:
import importlib.metadata

for pkg in ["pandas", "geopandas", "rasterio", "rioxarray", "xarray", "netCDF4", "shapely", "pyproj", "numpy", "matplotlib"]:
    print(pkg, importlib.metadata.version(pkg))